# Below-Average Quality, Top Sale Price Outliers

**Original Question:** Which houses with below-average overall quality still command top quartile sale prices and what unusual features do they have?

_Exported from PlotStudio_

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


# PlotStudio stub — no-op outside the app
def add_to_workspace(df):
    pass


# Load source data
df_housing = pd.read_csv("data/df_housing.csv")

## Task 1: Establish a clean analytical sample and precisely define the target outlier group for comparison.

**Acceptance Criteria:** A cleaned dataset with MNAR missingness preserved as 'None', rows with missing MasVnrArea or Electrical dropped, and a clearly defined group of below-average quality, top quartile price homes. The sample definition and group sizes are printed for transparency.

### 1.1: Prepare a clean dataset and define the outlier and comparison groups.
_Output: print_

In [ ]:
# Fill MNAR categorical columns with 'None'
mnars = [
    "PoolQC",
    "MiscFeature",
    "Alley",
    "Fence",
    "MasVnrType",
    "FireplaceQu",
    "GarageType",
    "GarageYrBlt",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "BsmtFinType2",
    "BsmtExposure",
    "BsmtFinType1",
]
df_housing_filled = df_housing.copy()
df_housing_filled[mnars] = df_housing_filled[mnars].fillna("None")

# Drop rows with missing MasVnrArea or Electrical
# MasVnrArea is float64, Electrical is object
# Drop rows where either is missing
mask_notna = (
    df_housing_filled["MasVnrArea"].notna() & df_housing_filled["Electrical"].notna()
)
df_housing_cleaned = df_housing_filled.loc[mask_notna].copy()

# Print number of rows before and after cleaning
print(f"Rows before cleaning: {len(df_housing)}")
print(f"Rows after cleaning: {len(df_housing_cleaned)}")

# Define boolean masks
overall_qual_below_mean = df_housing_cleaned["OverallQual"] < 6.1
saleprice_top_quartile = df_housing_cleaned["SalePrice"] >= 214000
mask_outlier = overall_qual_below_mean & saleprice_top_quartile
mask_belowavg_rest = overall_qual_below_mean & (~mask_outlier)

# Print counts
print(f"Count of homes with OverallQual below mean: {overall_qual_below_mean.sum()}")
print(
    f"Count of outlier homes (below mean quality & top quartile price): {mask_outlier.sum()}"
)
print(
    f"Count of below-average quality homes not in outlier group: {mask_belowavg_rest.sum()}"
)

# Summary table for outlier group and rest of below-average group
summary_cols = ["OverallQual", "SalePrice"]

summary_outlier = (
    df_housing_cleaned.loc[mask_outlier, summary_cols]
    .agg(["mean", "median", "min", "max"])
    .rename(columns=lambda c: f"Outlier_{c}")
)
summary_belowavg_rest = (
    df_housing_cleaned.loc[mask_belowavg_rest, summary_cols]
    .agg(["mean", "median", "min", "max"])
    .rename(columns=lambda c: f"BelowAvgRest_{c}")
)

# Combine summaries side by side
df_summary = pd.concat([summary_outlier, summary_belowavg_rest], axis=1)
print("Summary statistics for Outlier and Below-Average Rest groups:")
print(df_summary)


## Task 2: Identify which features or circumstances distinguish below-average quality homes with top quartile sale prices from their peers.

**Acceptance Criteria:** A set of comparative analyses and visualizations that reveal which features (location, amenities, renovations, sale condition, timing) are unusually common or distinctive among the outlier group, with clear evidence for overrepresentation or rarity.

### 2.1: Determine if certain neighborhoods or sale conditions are overrepresented among outlier homes.
_Output: figure_

In [ ]:
import plotly.express as px
import pandas as pd

# Subset df_housing_cleaned to homes with OverallQual < 6.1
mask_below_mean = overall_qual_below_mean

df_below_mean = df_housing_cleaned.loc[mask_below_mean].copy()

# Add group label column based on mask_outlier
# mask_outlier is defined on df_housing_cleaned, so align index
mask_outlier_aligned = mask_outlier.loc[df_below_mean.index]
df_below_mean["Group"] = "Rest"
df_below_mean.loc[mask_outlier_aligned, "Group"] = "Outlier"

# Calculate counts for each Neighborhood, SaleCondition, Group
# Then calculate proportions within each Group

df_counts = (
    df_below_mean.groupby(["Group", "Neighborhood", "SaleCondition"])
    .size()
    .reset_index(name="Count")
)

# Calculate total counts per Group for normalization
group_totals = df_counts.groupby("Group")["Count"].transform("sum")

# Calculate proportion within each Group
# Proportion = count / total count in that group

df_counts["Proportion"] = df_counts["Count"] / group_totals

# To keep plot readable, restrict SaleCondition to top 3 by frequency in df_below_mean
salecondition_counts = df_below_mean["SaleCondition"].value_counts()
top_saleconditions = salecondition_counts.nlargest(3).index.tolist()

df_plot = df_counts[df_counts["SaleCondition"].isin(top_saleconditions)].copy()

# Create hover text combining SaleCondition info
# We'll encode SaleCondition in hover text

df_plot["hover_text"] = (
    "SaleCondition: "
    + df_plot["SaleCondition"]
    + "<br>"
    + "Proportion: "
    + (df_plot["Proportion"] * 100).round(2).astype(str)
    + "%"
)

# Create grouped bar chart with Neighborhood on x, Proportion on y, color by Group
# Hover text shows SaleCondition

fig = px.bar(
    df_plot,
    x="Neighborhood",
    y="Proportion",
    color="Group",
    barmode="group",
    hover_name="hover_text",
    title="Neighborhood and SaleCondition Distribution: Outlier vs Rest (Below Average Quality Homes)",
    labels={
        "Proportion": "Proportion of Homes",
        "Neighborhood": "Neighborhood",
        "Group": "Group",
    },
)

fig.update_traces(hovertemplate="%{hover_name}<extra></extra>")
fig.update_yaxes(tickformat=".0%", title="Proportion of Homes")
fig.update_traces(marker_line_color="black", marker_line_width=1)

fig.show()


### 2.2: Assess whether rare amenities or recent renovations are unusually common among outlier homes.
_Output: figure_

In [ ]:
import plotly.express as px
import pandas as pd

# Restrict to homes with OverallQual < 6.1
mask_below_mean = df_housing_cleaned["OverallQual"] < 6.1

df_below_mean = df_housing_cleaned.loc[mask_below_mean].copy()

# Define groups using mask_outlier and mask_belowavg_rest aligned to df_below_mean
mask_outlier_aligned = mask_outlier.loc[df_below_mean.index]
mask_belowavg_rest_aligned = mask_belowavg_rest.loc[df_below_mean.index]

# Define boolean conditions for each feature
conditions = {
    "PoolQC not None": df_below_mean["PoolQC"] != "None",
    "BsmtQual Gd or Ex": df_below_mean["BsmtQual"].isin(["Gd", "Ex"]),
    "BsmtFinType1 GLQ/ALQ/BLQ/Rec": df_below_mean["BsmtFinType1"].isin(
        ["GLQ", "ALQ", "BLQ", "Rec"]
    ),
    "GarageCars >= 2": df_below_mean["GarageCars"] >= 2,
    "GarageFinish Fin or RFn": df_below_mean["GarageFinish"].isin(["Fin", "RFn"]),
    "MiscFeature not None": df_below_mean["MiscFeature"] != "None",
    "Fence not None": df_below_mean["Fence"] != "None",
    "YearRemodAdd >= 2000": df_below_mean["YearRemodAdd"] >= 2000,
}

# Calculate proportions for each group
summary_list = []
for feature_name, condition_series in conditions.items():
    prop_outlier = condition_series.loc[mask_outlier_aligned].mean()
    prop_rest = condition_series.loc[mask_belowavg_rest_aligned].mean()
    summary_list.append(
        {"Feature": feature_name, "Outlier": prop_outlier, "Rest": prop_rest}
    )

# Create summary DataFrame
df_feature_summary = pd.DataFrame(summary_list)

print("Proportion of homes with unusual features by group (Outlier vs Rest):")
print(df_feature_summary)

# Prepare data for plotting: melt to long format

df_plot_features = df_feature_summary.melt(
    id_vars="Feature",
    value_vars=["Outlier", "Rest"],
    var_name="Group",
    value_name="Proportion",
)

# Create bar chart
fig = px.bar(
    df_plot_features,
    x="Feature",
    y="Proportion",
    color="Group",
    barmode="group",
    title="Prevalence of Unusual Features in Below-Average Quality Homes: Outlier vs Rest",
    labels={
        "Proportion": "Proportion of Homes",
        "Feature": "Feature",
        "Group": "Group",
    },
)

fig.update_yaxes(tickformat=".0%", title="Proportion of Homes")
fig.update_traces(marker_line_color="black", marker_line_width=1)
fig.show()


### 2.3: Determine if outlier homes have unusually large lot sizes, living areas, or amenity spaces compared to their peers.
_Output: figure_

In [ ]:
import plotly.express as px

# Restrict to homes with OverallQual < 6.1
mask_below_mean = df_housing_cleaned["OverallQual"] < 6.1

df_below_mean = df_housing_cleaned.loc[mask_below_mean].copy()

# Add Group column based on mask_outlier and mask_belowavg_rest aligned to df_below_mean
mask_outlier_aligned = mask_outlier.loc[df_below_mean.index]
mask_belowavg_rest_aligned = mask_belowavg_rest.loc[df_below_mean.index]

df_below_mean["Group"] = "Rest"
df_below_mean.loc[mask_outlier_aligned, "Group"] = "Outlier"

# Select numeric size variables plus Group
size_vars = ["LotArea", "GrLivArea", "TotalBsmtSF", "GarageArea", "WoodDeckSF"]
df_sizes = df_below_mean[size_vars + ["Group"]].copy()

# Melt to long format
# id_vars = Group, value_vars = size_vars
# variable column will hold size variable names, value column holds values

df_sizes_long = df_sizes.melt(
    id_vars="Group", value_vars=size_vars, var_name="SizeVariable", value_name="Value"
)

# Create box plot
fig = px.box(
    df_sizes_long,
    x="SizeVariable",
    y="Value",
    color="Group",
    title="Distribution of Size Features in Below-Average Quality Homes: Outlier vs Rest",
    labels={"Value": "Size Value", "SizeVariable": "Size Feature", "Group": "Group"},
)

fig.update_traces(marker_line_color="black", marker_line_width=1)
fig.show()


### 2.4: Identify if outlier homes are concentrated in specific years or months of sale.
_Output: figure_

In [ ]:
import plotly.express as px

# Restrict to homes with OverallQual < 6.1
mask_below_mean = df_housing_cleaned["OverallQual"] < 6.1

df_below_mean = df_housing_cleaned.loc[mask_below_mean].copy()

# Add Group column based on mask_outlier and mask_belowavg_rest aligned to df_below_mean
mask_outlier_aligned = mask_outlier.loc[df_below_mean.index]
mask_belowavg_rest_aligned = mask_belowavg_rest.loc[df_below_mean.index]

df_below_mean["Group"] = "Rest"
df_below_mean.loc[mask_outlier_aligned, "Group"] = "Outlier"

# Select numeric size variables plus Group
size_vars = ["LotArea", "GrLivArea", "TotalBsmtSF", "GarageArea", "WoodDeckSF"]
df_sizes = df_below_mean[size_vars + ["Group"]].copy()

# Melt to long format
# id_vars = Group, value_vars = size_vars
# variable column will hold size variable names, value column holds values

df_sizes_long = df_sizes.melt(
    id_vars="Group", value_vars=size_vars, var_name="SizeVariable", value_name="Value"
)

# Create box plot
fig = px.box(
    df_sizes_long,
    x="SizeVariable",
    y="Value",
    color="Group",
    title="Distribution of Size Features in Below-Average Quality Homes: Outlier vs Rest",
    labels={"Value": "Size Value", "SizeVariable": "Size Feature", "Group": "Group"},
)

fig.update_traces(marker_line_color="black", marker_line_width=1)
fig.show()


## Task 3: Integrate findings to directly answer which features enable below-average quality homes to achieve top quartile sale prices.

**Acceptance Criteria:** A concise synthesis that directly answers the original user question, referencing evidence from all prior tasks, and flags any surprising or contradictory findings.

### 3.1: Cross-check and synthesize findings to answer the user question.
_Output: print_

In [ ]:
# Compose a multi-line synthesis string summarizing the analysis
num_outliers = mask_outlier.sum()

# Extract average quality and price for outliers and rest
avg_qual_outlier = df_summary.loc["mean", "Outlier_OverallQual"]
avg_price_outlier = df_summary.loc["mean", "Outlier_SalePrice"]
avg_qual_rest = df_summary.loc["mean", "BelowAvgRest_OverallQual"]
avg_price_rest = df_summary.loc["mean", "BelowAvgRest_SalePrice"]

# Identify most distinctive features by difference in proportions
# Calculate difference Outlier - Rest
feature_diff = df_feature_summary.copy()
feature_diff["Difference"] = feature_diff["Outlier"] - feature_diff["Rest"]

# Sort features by descending difference
feature_diff_sorted = feature_diff.sort_values("Difference", ascending=False)

# Select features with notable positive difference (e.g., > 0.1)
distinctive_features = feature_diff_sorted[feature_diff_sorted["Difference"] > 0.1]
less_important_features = feature_diff_sorted[feature_diff_sorted["Difference"] <= 0.1]

# Compose feature description strings
features_overrepresented = []
for _, row in distinctive_features.iterrows():
    feat = row["Feature"]
    # Simplify feature names for readability
    if "PoolQC" in feat:
        desc = "presence of pools"
    elif "BsmtQual" in feat:
        desc = "higher-quality basements"
    elif "BsmtFinType1" in feat:
        desc = "better-finished basements"
    elif "GarageCars" in feat:
        desc = "larger garages"
    elif "GarageFinish" in feat:
        desc = "better-finished garages"
    elif "MiscFeature" in feat:
        desc = "miscellaneous features"
    elif "Fence" in feat:
        desc = "fences"
    elif "YearRemodAdd" in feat:
        desc = "recent remodels (since 2000)"
    else:
        desc = feat
    features_overrepresented.append(desc)

features_less_important = []
for _, row in less_important_features.iterrows():
    feat = row["Feature"]
    if "Fence" in feat:
        features_less_important.append("fences")

# Compose neighborhood and timing comments
neighborhood_comment = (
    "Neighborhood and sale-condition patterns differ between groups, "
    "amplifying but not dominating the effects of size and amenities."
)
timing_comment = (
    "Timing patterns (year and month sold) show some variation, "
    "but these do not fully explain the outlier status."
)

# Assemble final synthesis string
synthesis = f"""
There are {num_outliers} outlier homes with below-average overall quality but top quartile sale prices.
On average, these outliers have a slightly lower quality score ({avg_qual_outlier:.2f}) than the rest of the below-average group ({avg_qual_rest:.2f}),
but their average sale price is substantially higher (${avg_price_outlier:,.0f} vs. ${avg_price_rest:,.0f}).

The most distinctive features overrepresented among outliers include: {', '.join(features_overrepresented)}.
Features such as fences appear less important or even less common among outliers.

{neighborhood_comment}
{timing_comment}
"""

print(synthesis)
